# MiniMax H3 — 인물 사진을 자연스러운 표정 영상으로 (Kaggle T4 x2)

인물 사진 한 장을 넣으면, 같은 얼굴과 포즈에서 시작해서 **자연스러운 표정 변화, 고개 움직임, 손짓**이 있는
클로즈업 영상을 만들어요. 원본 워크플로우 `033.REALminimax_h3-Upscale`을 T4에 맞게 옮긴 버전이에요.

**원본 워크플로우의 흐름:** 사진을 레퍼런스이자 첫 프레임 가이드로 넣음 → 8스텝(er_sde, beta, 시프트 6/3)
→ 잠재 업스케일 2배 + 4스텝 다듬기 → FILM 보간으로 48fps

**T4 기준 기본값:** 0.2MP, 3초, 업스케일 끔, FILM 보간 켬.
영상 모델은 Singularity(얼굴 왜곡과 피부 번들거림을 줄인 미세조정 모델)를 써요. 원본은 Comfy 공식 ref2va 모델이에요.

**준비 (Kaggle 노트북 오른쪽 패널)**
1. Settings → Accelerator: **GPU T4 x2**
2. Settings → Internet: **On**
3. Add Input → 이미지를 올린 **Dataset**을 추가하고, 설정 셀의 경로를 맞춰요.
4. (선택) Add-ons → Secrets에 `HF_TOKEN`을 추가하면 다운로드가 안정적이에요.

**실행:** 위에서부터 순서대로 실행해요. 결과 영상은 `/kaggle/working/outputs`에 저장돼요.
설정만 바꿔 다시 만들 때는 설정 셀과 **5번 셀만** 다시 실행하면 돼요(모델과 서버는 그대로 재사용).

In [ ]:
# ============ 설정 (여기만 고치면 돼요) ============

# 영상 모델: "gguf" (T4 권장, 어떤 GPU에서도 동작) | "w4a8" (11.8GB, T4 지원 불확실) | "int8_pruned" (21GB)
MODEL_VARIANT = "gguf"
GGUF_PREFERENCE = ["Q4_K_M", "Q4_K_S", "Q4_0", "Q5_K_M", "Q3_K_M"]  # 앞에 있는 것부터 찾아요

# 텍스트 인코더: "nvfp4_awq" (약 15GB, T4 권장) | "int8_convrot" (약 25GB, T4에 너무 커요)
TEXT_ENCODER = "nvfp4_awq"
TE_DEVICE = "cuda:1"          # 텍스트 인코더를 두 번째 T4에 올려요. 문제가 생기면 "default"

TURBO_LORA = False            # 원본은 LoRA 없이 8스텝이에요. 결과가 흐리면 True로 (Turbo 4스텝 LoRA)

INPUT_IMAGE = "/kaggle/input/my-photos/portrait.jpg"   # 인물 사진 한 장

PROMPT_MODE = "both"          # "both": 표정+카메라 지시문 | "camera": 원본 그대로(카메라만) | "custom"
CUSTOM_PROMPT = ""            # PROMPT_MODE="custom"일 때 사용

SECONDS = 3.0                 # 영상 길이(초). 원본은 8초, T4에서는 2~4초 권장
MEGAPIXELS = 0.2              # 원본은 0.3
ASPECT = "auto"               # "auto": 사진 비율에 가장 가까운 것 | "16:9" | "9:16" | "1:1"
FPS = 24
STEPS = 8
SAMPLER = "er_sde"            # 원본 설정. TURBO_LORA=True면 "euler" 권장
SHIFT = (6.0, 3.0)            # MiniMaxH3SigmaShift (영상, 음성). 원본 설정
SEED = 835

INTERPOLATE = True            # FILM 2배 프레임 보간 → 48fps (가벼워요)
UPSCALE = False               # 원본의 잠재 업스케일 + 2차 다듬기. T4에서는 메모리 부족 가능성이 커요
UPSCALE_FACTOR = 1.5          # 원본은 2.0
PASS2_SIGMAS = "0.9035, 0.8000, 0.6316, 0.3158, 0.0000"   # 원본 2차 시그마

EXTRA_NODE_PACKS = {"Comfyui_Minimax_h3_latent_Upscaler": "https://github.com/LBH-123-AI/Comfyui_Minimax_h3_latent_Upscaler"} if UPSCALE else {}
INPUT_FILES = [INPUT_IMAGE]

# T4는 bf16을 지원하지 않아서 fp16으로 강제해요. 영상이 검게 나오면 "--force-fp32"로 바꿔 보세요(느려요).
DTYPE_FLAG = "--force-fp16"
COMFY_ARGS = ["--lowvram", "--disable-smart-memory", "--cache-none", "--reserve-vram", "1.0", DTYPE_FLAG]

COMFY = "/tmp/ComfyUI"
MODEL_DIR = "/tmp/models"
OUT_DIR = "/kaggle/working/outputs"

### 프롬프트

In [ ]:
# 원본 워크플로우의 지시문 두 개예요. 원본에서는 카메라 지시문만 연결되어 있었어요.
EXPRESSION_PROMPT = """### Expressive Facial and Gesture Animation

Give the existing subject natural, expressive, and visually appealing facial and upper-body animation while strictly preserving the original pose, identity, anatomy, clothing, and scene.

The face should not remain expressionless or frozen.

Introduce a natural progression of varied facial expressions throughout the video, selected according to the subject's appearance, pose, and context.

Possible expressions include subtle:

* gentle smile
* bright smile
* playful smile
* cute smile
* shy smile
* mischievous expression
* curious expression
* slightly surprised expression
* soft amused expression
* confident expression
* flirtatious expression
* charming expression
* relaxed expression
* momentary serious expression followed by a smile
* subtle eyebrow raise
* playful eyebrow movement
* gentle eye widening
* soft narrowing of the eyes
* natural blinking
* brief eye contact with the camera
* looking slightly away and then back toward the camera
* small changes in mouth shape
* subtle lip movement
* slight cheek movement
* natural facial muscle movement

Expressions should transition smoothly rather than changing abruptly.

Do not keep the same facial expression for the entire video.

Use several complementary expressions with natural timing and variation rather than repeating one expression continuously.

The expressions should remain believable and physically natural, never exaggerated into a cartoon-like performance.

### Head and Neck Movement

Allow the subject to naturally move the head while maintaining the original body position.

Possible movements include:

* gently lifting the head
* slightly lowering the head
* turning the head subtly left
* turning the head subtly right
* tilting the head slightly left
* tilting the head slightly right
* returning the head naturally toward center
* briefly looking upward
* briefly looking downward
* turning toward the camera
* looking away and naturally returning
* a small playful head tilt
* subtle nodding
* gentle repositioning of the neck and head

Head movement must remain physically compatible with the original pose.

If the subject is lying down, keep the subject lying down.

If the subject is sitting, keep the subject sitting.

If the subject is standing, keep the subject standing.

Never turn a lying subject into a standing or walking subject unless the reference image clearly supports the complete physical transition.

Head movement should be expressive but restrained, with realistic neck mechanics.

### Natural Hand and Arm Gestures

When hands or arms are visible and their position allows movement, introduce natural expressive hand gestures.

Possible gestures include:

* gently raising one hand
* small finger movements
* subtle finger repositioning
* lightly touching the hair
* gently brushing or adjusting the hair
* touching the cheek
* briefly touching the chin
* resting a hand naturally near the face
* small playful wave
* gentle hand gesture toward the camera
* lightly adjusting clothing that is already present
* moving the wrist naturally
* opening and relaxing the fingers
* subtle movement of the forearm
* briefly placing one hand over another
* moving the hand slightly closer to or farther from the body

Use only gestures physically supported by the original image.

Do not invent a hand that is not visible.

Do not create extra fingers, extra hands, duplicated arms, or anatomically impossible movements.

Keep the original number of limbs and preserve hand anatomy throughout the entire video.

### Cute, Playful, Charming and Flirtatious Behavior

When appropriate to the subject and reference image, allow subtle changes in personality and expression that make the subject feel alive.

The subject may naturally alternate between:

a soft cute expression → gentle smile → playful look → brief eye contact → slight head tilt → relaxed smile.

Other natural combinations may include:

a neutral expression → subtle smile → playful eyebrow raise → slight head tilt → soft smile.

Or:

a relaxed expression → look toward camera → small smile → look away → return eye contact with a charming expression.

For a more confident and attractive presentation, expressions may include a subtle confident smile, playful eye contact, slight head tilt, relaxed posture adjustment, or a gentle hair-touching gesture.

Keep all expressions tasteful, natural, and photorealistic.

Do not exaggerate facial expressions.

Do not create theatrical acting.

Do not make the subject repeatedly perform the same gesture.

### Coordinated Facial + Head + Hand Animation

Facial expressions, head movement, eye movement, and hand gestures should feel connected rather than randomly animated.

For example:

A slight smile may naturally accompany a gentle head tilt.

A curious expression may accompany a subtle eyebrow raise and slight head turn.

A playful expression may accompany brief eye contact and a small hand gesture.

A shy smile may accompany looking slightly away before returning attention toward the camera.

A confident expression may accompany a subtle posture adjustment and relaxed hand movement.

Use these combinations selectively and naturally.

Do not perform every possible action in a single video.

Choose several complementary movements based on what is actually visible in the reference image.

### Expression Timing

Create gradual expression changes throughout the shot.

Avoid maintaining one static facial expression from beginning to end.

Use natural pauses between expressions.

Allow brief neutral moments between stronger expressions so the animation feels human rather than continuously performed.

A possible progression is:

Beginning:
preserve the exact reference expression and pose.

Early movement:
natural blinking, subtle eye movement, small facial muscle changes.

Middle:
gentle smile or playful expression, followed by a small head movement.

Later:
slight head tilt, brief eye contact, subtle hand or hair gesture when physically possible.

Ending:
settle naturally into a relaxed, charming expression while maintaining the same identity and pose.

The exact sequence should adapt automatically to the reference image rather than being rigidly forced.

### Important Pose Constraint

Expressiveness must NEVER override physical plausibility.

The subject may become more expressive, but must remain in the same fundamental physical situation shown in the reference image.

A lying subject remains lying.

A seated subject remains seated.

A standing subject remains standing.

Do not force walking, standing, sitting, lying down, dancing, or large body movements unless the original reference pose clearly supports such movement.

Prioritize expressive face, eyes, head, hands, fingers, hair, and small posture adjustments before attempting large body movements.

The result should feel like a real person naturally reacting, smiling, looking around, tilting their head, and making small gestures while being filmed continuously.
"""

CAMERA_PROMPT = """### Close Cinematic Camera Movement

Keep the camera relatively close to the subject throughout the video.

Prefer:

* close-up
* medium close-up
* upper-body framing
* intimate portrait framing

Avoid unnecessarily wide or distant shots.

Do not make the subject appear small in the frame.

The face, eyes, expression, hair, hands, and upper-body details should remain clearly visible whenever possible.

Select only ONE smooth primary camera movement for each video.

Possible camera movements include:

* gentle push-in
* very subtle pull-back
* slow left-to-right camera drift
* slow right-to-left camera drift
* subtle lateral tracking
* gentle forward arc
* gentle side arc
* slight upward reframing
* slight downward reframing
* very subtle partial orbit
* soft handheld-style micro movement while remaining stable

Choose the camera movement according to the subject's pose, face direction, composition, and available space.

Do not always move the camera in the same direction.

For different generations, naturally vary between left movement, right movement, subtle push-in, gentle arc movement, or slight vertical reframing.

However, camera variety must remain subtle and controlled.

The camera should enhance the subject's facial expression and gesture rather than becoming the main action.

For facial expressions or eye contact:

prefer a gentle push-in or subtle lateral drift.

For a head tilt or head turn:

prefer a slow lateral movement or very gentle arc.

For touching the hair, cheek, or chin:

prefer close framing with a subtle push-in or side drift.

For small hand gestures near the face:

keep both the face and hand clearly visible within the frame.

For a lying subject:

keep the camera close and move gently around the existing angle without revealing large unseen areas of the environment.

For a seated subject:

favor close or medium-close portrait framing with subtle lateral or forward camera movement.

For a standing subject:

still prioritize upper-body or medium-close framing rather than automatically showing the full body.

Do not automatically zoom out simply because the subject is standing.

Avoid:

* full-body wide shots unless absolutely necessary
* distant framing
* dramatic zoom-out
* aggressive orbiting
* fast camera movement
* rapid left-right motion
* large vertical camera movement
* excessive camera shake
* sudden perspective changes
* revealing large unsupported areas outside the reference image
* repeatedly moving closer and farther away

The camera should feel smooth, intimate, cinematic, and physically realistic.

Maintain stable facial detail and subject identity throughout the camera movement.

Use natural parallax without distorting the face, body, background, or perspective.

The camera should generally remain close enough that the subject occupies a large and visually important portion of the frame.

Prioritize facial quality and expressive detail over showing the entire body.

The final result should feel like a skilled camera operator gently filming the subject from close range while the subject naturally changes expression, moves the head, and performs subtle hand or upper-body gestures.
"""

PROMPT = {"both": EXPRESSION_PROMPT + "\n\n" + CAMERA_PROMPT, "camera": CAMERA_PROMPT, "custom": CUSTOM_PROMPT}[PROMPT_MODE]
print(f"프롬프트 {len(PROMPT)}자 ({PROMPT_MODE})")

## 1. 환경 확인

In [ ]:
import os, shutil, subprocess

subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"])
total, used, free = shutil.disk_usage("/tmp")
print(f"/tmp 여유 공간: {free / 1e9:.0f} GB (모델에 약 30GB 필요)")
with open("/proc/meminfo") as f:
    mem = {l.split(":")[0]: int(l.split()[1]) for l in f}
print(f"RAM: {mem['MemTotal'] / 1e6:.0f} GB, 사용 가능 {mem['MemAvailable'] / 1e6:.0f} GB")
if free < 32e9:
    print("⚠ /tmp 공간이 부족할 수 있어요. 다운로드가 실패하면 MODEL_VARIANT를 더 작은 것으로 바꾸세요.")
missing = [p for p in INPUT_FILES if not os.path.isfile(p)]
if missing:
    print("⚠ 입력 이미지를 찾을 수 없어요:", missing)
    print("  /kaggle/input 안의 파일:")
    for root, _, files in os.walk("/kaggle/input"):
        for fn in files[:20]:
            print("   ", os.path.join(root, fn))

## 2. ComfyUI와 커스텀 노드 설치

In [ ]:
import subprocess, sys, os

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if not os.path.isdir(COMFY):
    sh(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY}")
else:
    sh(f"git -C {COMFY} pull --ff-only")

# 검증된 노드 팩만 사용해요 (출처가 불분명한 노드 팩은 설치하지 않아요)
NODE_PACKS = {
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes",      # MiniMax 저VRAM 패치
    "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF",           # GGUF 모델 로더
    "ComfyUI-MultiGPU": "https://github.com/pollockjj/ComfyUI-MultiGPU", # 텍스트 인코더를 두 번째 GPU로
    **EXTRA_NODE_PACKS,
}
for name, url in NODE_PACKS.items():
    path = f"{COMFY}/custom_nodes/{name}"
    if not os.path.isdir(path):
        sh(f"git clone --depth 1 {url} {path}")

sh(f"{sys.executable} -m pip install -q -r {COMFY}/requirements.txt")
for name in NODE_PACKS:
    req = f"{COMFY}/custom_nodes/{name}/requirements.txt"
    if os.path.isfile(req):
        sh(f"{sys.executable} -m pip install -q -r {req}")
sh(f"{sys.executable} -m pip install -q -U huggingface_hub gguf")

import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.device_count())

## 3. 모델 다운로드 (`/tmp/models`, 세션이 끝나면 지워져요)

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

api = HfApi(token=token)

def pick(repo, includes, prefer=(), ext=".safetensors"):
    files = [f for f in api.list_repo_files(repo) if f.endswith(ext)]
    hits = [f for f in files if all(s.lower() in f.lower() for s in includes)]
    for p in prefer:
        for f in hits:
            if p.lower() in f.lower():
                return f
    if hits:
        return hits[0]
    raise FileNotFoundError(f"{repo}에서 {includes} 파일을 못 찾았어요. 파일 목록:\n  " + "\n  ".join(files))

SINGULARITY = "WarmBloodAban/Minimax-h3_Singularity"
SINGULARITY_GGUF = "Abiray/MiniMax-H3-Singularity-GGUF"
COMFY_ORG = "Comfy-Org/MiniMax-H3"
TURBO = "lightx2v/Minimax-h3-Turbo"

plan = {}  # 종류 -> (저장소, 파일, ComfyUI 모델 폴더)
if MODEL_VARIANT == "gguf":
    plan["unet"] = (SINGULARITY_GGUF, pick(SINGULARITY_GGUF, [], GGUF_PREFERENCE, ext=".gguf"), "unet")
elif MODEL_VARIANT == "w4a8":
    plan["unet"] = (SINGULARITY, pick(SINGULARITY, ["w4a8"]), "diffusion_models")
elif MODEL_VARIANT == "int8_pruned":
    plan["unet"] = (SINGULARITY, pick(SINGULARITY, ["pruned", "int8"]), "diffusion_models")
else:
    raise ValueError(MODEL_VARIANT)
te_key = {"nvfp4_awq": ["qwen3vl", "nvfp4"], "int8_convrot": ["qwen3vl", "int8"]}[TEXT_ENCODER]
plan["te"] = (COMFY_ORG, pick(COMFY_ORG, te_key), "text_encoders")
plan["vae"] = (COMFY_ORG, pick(COMFY_ORG, ["video_vae", "fp16"]), "vae")
plan["avae"] = (COMFY_ORG, pick(COMFY_ORG, ["audio_vae"]), "vae")
if TURBO_LORA:
    plan["lora"] = (TURBO, pick(TURBO, ["ref2v", "4step", "comfyui"], ["v0.1"]), "loras")
if INTERPOLATE:
    plan["film"] = ("Comfy-Org/frame_interpolation", pick("Comfy-Org/frame_interpolation", ["film_net", "fp16"]), "frame_interpolation")
if UPSCALE:
    plan["upscaler"] = ("LBH-123-AI/Minimax_h3_latent_Upscaler",
                        pick("LBH-123-AI/Minimax_h3_latent_Upscaler", ["latent_upscaler"], ["fp16", "bf16"]),
                        "latent_upscale_models")

FILES = {}
for key, (repo, fname, folder) in plan.items():
    print(f"⬇ {key}: {repo}/{fname}")
    local = hf_hub_download(repo, fname, local_dir=f"{MODEL_DIR}/{repo.replace('/', '__')}", token=token)
    os.makedirs(f"{COMFY}/models/{folder}", exist_ok=True)
    link = f"{COMFY}/models/{folder}/{os.path.basename(fname)}"
    if not os.path.exists(link):
        os.symlink(local, link)
    FILES[key] = os.path.basename(fname)
    print(f"   {os.path.getsize(local) / 1e9:.1f} GB → models/{folder}/{FILES[key]}")
print(FILES)

## 4. ComfyUI 서버 시작

In [ ]:
import subprocess, sys, time, json, urllib.request

BASE = "http://127.0.0.1:8188"
LOG = "/tmp/comfyui.log"

def server_up():
    try:
        urllib.request.urlopen(BASE + "/system_stats", timeout=5)
        return True
    except Exception:
        return False

if not server_up():
    server = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", *COMFY_ARGS],
        cwd=COMFY, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    )
    for _ in range(180):
        if server_up() or server.poll() is not None:
            break
        time.sleep(5)

def log_tail(n=40):
    with open(LOG, errors="replace") as f:
        print("".join(f.readlines()[-n:]))

if server_up():
    print("✅ ComfyUI 실행 중")
    log_tail(15)
else:
    print("❌ ComfyUI가 시작되지 않았어요. 로그:")
    log_tail(80)

## 5. 워크플로우 도구 준비

In [ ]:
import json, math, os, shutil, time, uuid, urllib.request, urllib.error

def api(path, data=None):
    body = json.dumps(data).encode() if data is not None else None
    req = urllib.request.Request(BASE + path, data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())

INFO = api("/object_info")

def combo_options(spec):
    t = spec[0]
    if isinstance(t, list):
        return t
    if t == "COMBO" and len(spec) > 1 and isinstance(spec[1], dict):
        return spec[1].get("options", [])
    return None

class Graph:
    """ComfyUI API 형식 그래프. 서버의 노드 정의로 입력 이름과 값을 검사해요."""
    def __init__(self):
        self.nodes, self.n = {}, 0

    def add(self, cls, **inputs):
        if cls not in INFO:
            raise RuntimeError(f"서버에 '{cls}' 노드가 없어요. 설치 셀 로그를 확인하세요.")
        req = INFO[cls]["input"].get("required", {})
        specs = {**req, **INFO[cls]["input"].get("optional", {})}
        for name, spec in req.items():
            if name in inputs or any(k.startswith(name + ".") for k in inputs):
                continue
            cfg = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}
            opts = combo_options(spec)
            if opts:
                inputs[name] = cfg.get("default", opts[0])
            elif "default" in cfg:
                inputs[name] = cfg["default"]
            else:
                raise RuntimeError(f"{cls}: 필수 입력 '{name}'이 비어 있어요. 정의: {spec}")
        for name, value in inputs.items():
            if name not in specs:
                print(f"  · {cls}: 정의에 없는 입력 '{name}' (동적 입력이면 정상)")
                continue
            opts = combo_options(specs[name])
            if opts and isinstance(value, str) and value not in opts:
                raise RuntimeError(f"{cls}.{name}='{value}'가 선택지에 없어요: {opts[:15]}")
        self.n += 1
        nid = str(self.n)
        self.nodes[nid] = {"class_type": cls, "inputs": inputs}
        return nid

def out(nid, i=0):
    return [nid, i]

def video_size(aspect, megapixels):
    """원본 워크플로우의 해상도 표와 같은 크기 (32의 배수)."""
    ar = {"16:9": 16 / 9, "9:16": 9 / 16, "1:1": 1.0}[aspect]
    w = math.ceil(math.sqrt(megapixels * 1e6 * ar) / 32) * 32
    h = math.ceil(math.sqrt(megapixels * 1e6 / ar) / 32) * 32
    return w, h

def frame_count(seconds):
    """원본 워크플로우의 프레임 수 계산식."""
    f = max(5, round(seconds * 24))
    return f + (5 - f % 17) % 17

def add_input_file(g, path, name, node="LoadImage", field="image"):
    os.makedirs(f"{COMFY}/input", exist_ok=True)
    fname = f"{name}{os.path.splitext(path)[1]}"
    shutil.copy(path, f"{COMFY}/input/{fname}")
    INFO[node] = api(f"/object_info/{node}")[node]  # 새 파일이 선택지에 보이도록 갱신
    return out(g.add(node, **{field: fname}))

def add_input_image(g, path, name):
    return add_input_file(g, path, name)

def load_models(g, shift=None):
    """영상 모델, 텍스트 인코더, VAE를 불러오고 T4용 패치와 LoRA를 적용해요."""
    if MODEL_VARIANT == "gguf":
        model = out(g.add("UnetLoaderGGUF", unet_name=FILES["unet"]))
    else:
        model = out(g.add("UNETLoader", unet_name=FILES["unet"], weight_dtype="default"))
    if TE_DEVICE != "default" and "CLIPLoaderMultiGPU" in INFO:
        clip = out(g.add("CLIPLoaderMultiGPU", clip_name=FILES["te"], type="minimax", device=TE_DEVICE))
    else:
        if TE_DEVICE != "default":
            print("⚠ CLIPLoaderMultiGPU가 없어서 텍스트 인코더를 기본 GPU에 올려요.")
        clip = out(g.add("CLIPLoader", clip_name=FILES["te"], type="minimax"))
    vae = out(g.add("VAELoader", vae_name=FILES["vae"]))
    audio_vae = out(g.add("VAELoader", vae_name=FILES["avae"]))
    # 원본의 SageAttention/SolAttention은 T4에서 지원되지 않아 KJNodes 저VRAM 패치로 대신해요.
    if "MiniMaxChunkFeedForward" in INFO:
        model = out(g.add("MiniMaxChunkFeedForward", model=model, chunks=4, seq_threshold=2048))
    if "MiniMaxLowVRAMAttention" in INFO:
        model = out(g.add("MiniMaxLowVRAMAttention", model=model, head_chunks=4))
    if shift:
        model = out(g.add("MiniMaxH3SigmaShift", model=model, shift_video=shift[0], shift_audio=shift[1]))
    if TURBO_LORA:
        lora = g.add("LoraLoader", model=model, clip=clip, lora_name=FILES["lora"],
                     strength_model=1.0, strength_clip=1.0)
        model, clip = out(lora, 0), out(lora, 1)
    return model, clip, vae, audio_vae

def save_video(g, images, audio, fps, prefix):
    video = out(g.add("CreateVideo", images=images, fps=float(fps), audio=audio))
    g.add("SaveVideo", video=video, filename_prefix=prefix)

def run(g):
    with open("/kaggle/working/last_workflow_api.json", "w") as fp:
        json.dump(g.nodes, fp, indent=1, ensure_ascii=False)
    try:
        prompt_id = api("/prompt", {"prompt": g.nodes, "client_id": uuid.uuid4().hex})["prompt_id"]
    except urllib.error.HTTPError as e:
        print("❌ 워크플로우 오류:")
        print(json.dumps(json.loads(e.read()), indent=1, ensure_ascii=False)[:4000])
        raise
    start = time.time()
    while True:
        hist = api(f"/history/{prompt_id}")
        if prompt_id in hist:
            status = hist[prompt_id].get("status", {})
            if status.get("status_str") == "error":
                print("\n❌ 실행 중 오류:")
                for m in status.get("messages", []):
                    if m[0] == "execution_error":
                        print(m[1].get("node_type"), "-", m[1].get("exception_message"))
                log_tail(40)
                raise RuntimeError("generation failed")
            break
        time.sleep(15)
        lines = open(LOG, errors="replace").read().replace("\r", "\n").strip().splitlines()
        print(f"\r{(time.time() - start) / 60:.1f}분 경과 | {lines[-1][:110] if lines else ''}", end="", flush=True)
    print(f"\n✅ 완료: {(time.time() - start) / 60:.1f}분")

print(f"노드 정의 {len(INFO)}개 불러옴")

## 6. 영상 만들기

In [ ]:
from PIL import Image

aspect = ASPECT
if aspect == "auto":
    with Image.open(INPUT_IMAGE) as im:
        r = im.width / im.height
    aspect = min({"16:9": 16 / 9, "9:16": 9 / 16, "1:1": 1.0}.items(), key=lambda kv: abs(math.log(r / kv[1])))[0]
W, H = video_size(aspect, MEGAPIXELS)
LENGTH = frame_count(SECONDS)
print(f"비율 {aspect}, 해상도 {W}x{H}, {LENGTH}프레임")

g = Graph()
model, clip, vae, audio_vae = load_models(g, shift=SHIFT)

photo = add_input_image(g, INPUT_IMAGE, "face_0")
# 첫 프레임 가이드용: 영상 크기에 맞게 가운데를 잘라 맞춰요 (원본의 ImageResizeKJv2 crop/center와 같은 역할)
guide = out(g.add("ImageScale", image=photo, upscale_method="lanczos", width=W, height=H, crop="center"))

r2v = g.add("MiniMaxH3ReferenceToVideo", clip=clip, vae=vae, audio_vae=audio_vae,
            prompt=PROMPT, width=W, height=H, length=LENGTH, **{"ref_images.ref_image_0": photo})
positive = out(g.add("MiniMaxH3AddGuide", positive=out(r2v, 0), latent=out(r2v, 1), vae=vae,
                     audio_vae=audio_vae, image=guide, frame_idx=0))

noise = out(g.add("RandomNoise", noise_seed=SEED))
sampler = out(g.add("KSamplerSelect", sampler_name=SAMPLER))
sched = out(g.add("BasicScheduler", model=model, scheduler="beta", steps=STEPS, denoise=1.0))
guider = out(g.add("BasicGuider", model=model, conditioning=positive))
s1 = g.add("SamplerCustomAdvanced", noise=noise, guider=guider, sampler=sampler,
           sigmas=sched, latent_image=out(r2v, 1))
latent = out(s1, 0)

if UPSCALE:
    av = g.add("LTXVSeparateAVLatent", av_latent=latent)
    up = g.add("MinimaxH3LatentUpscaler3D", latent=out(av, 0), model_name=FILES["upscaler"],
               mode="scale by multiplier", **{"mode.scale": UPSCALE_FACTOR}, device="cuda", precision="fp16")
    merged = out(g.add("LTXVConcatAVLatent", video_latent=out(up), audio_latent=out(av, 1)))
    sigmas2 = out(g.add("ManualSigmas", sigmas=PASS2_SIGMAS))
    # 원본처럼 2차에서는 첫 프레임 가이드 없이 레퍼런스 조건만 써요
    guider2 = out(g.add("BasicGuider", model=model, conditioning=out(r2v, 0)))
    s2 = g.add("SamplerCustomAdvanced", noise=noise, guider=guider2, sampler=sampler,
               sigmas=sigmas2, latent_image=merged)
    latent = out(s2, 0)

images = out(g.add("VAEDecode", samples=latent, vae=vae))
audio = out(g.add("VAEDecodeAudio", samples=latent, vae=audio_vae))
fps = FPS
if INTERPOLATE:
    film = out(g.add("FrameInterpolationModelLoader", model_name=FILES["film"]))
    images = out(g.add("FrameInterpolate", interp_model=film, images=images, multiplier=2))
    fps = FPS * 2

PREFIX = f"face/face_{int(time.time())}"
save_video(g, images, audio, fps, PREFIX)
run(g)

## 7. 결과 보기

In [ ]:
import glob, os, shutil
from IPython.display import Video, display

os.makedirs(OUT_DIR, exist_ok=True)
outs = sorted(glob.glob(f"{COMFY}/output/{PREFIX}*"), key=os.path.getmtime)
for path in outs:
    dest = shutil.copy(path, OUT_DIR)
    print("저장:", dest)
    display(Video(dest, embed=True, width=640))
if not outs:
    print("결과 파일이 없어요. 로그:")
    log_tail(40)

## 문제 해결

오류가 나면 **오류 메시지와 로그 마지막 부분**을 그대로 복사해서 Claude에게 붙여 주세요.
실행한 워크플로우는 `/kaggle/working/last_workflow_api.json`에 저장돼요.

| 증상 | 시도해 볼 것 |
|---|---|
| `CUDA out of memory` | `SECONDS`나 `MEGAPIXELS`를 낮추기, `GGUF_PREFERENCE`를 `Q3_K_M` 우선으로 |
| 커널이 죽음 (RAM 부족) | `TE_DEVICE="cuda:1"` 유지, 더 작은 GGUF 사용 |
| 영상이 검정/노이즈 | `DTYPE_FLAG="--force-fp32"` (느려짐) |
| `서버에 '...' 노드가 없어요` | 2번 셀 로그에서 해당 노드 팩 설치 오류 확인 |
| 텍스트 인코더 단계가 매우 느림 | NVFP4는 T4에서 에뮬레이션이라 느려요. 정상일 수 있어요 |